# 📋 claude.ai 프롬프트 / claude.ai Prompt

> **수업 활용법:** 아래 프롬프트를 [claude.ai](https://claude.ai)에 붙여넣으면 유사한 노트북을 생성할 수 있습니다.

---

### 🇰🇷 한국어 프롬프트
```
Anthropic Claude API와 sentence-transformers를 이용한 간단한 RAG 파이프라인
Google Colab 노트북을 한국어로 작성해주세요.
다음 내용을 포함하세요:
1. 필요한 라이브러리 설치 (anthropic, sentence-transformers, numpy)
2. 가상 회사 'TechKorea' HR 정책 텍스트 6개 단락으로 지식 베이스 구성
   (연차, 재택근무, 복리후생, 장비지원, 경비처리, 교육지원 관련)
3. sentence-transformers로 임베딩 생성 후 numpy 배열로 저장
4. 질문 임베딩 → 코사인 유사도로 상위 2개 단락 검색하는 함수 구현
5. 검색된 단락을 컨텍스트로 Claude claude-sonnet-4-6에 전달하여 답변 생성
6. 같은 질문에 대해 'RAG 없음 vs RAG 있음' 비교 출력
7. 미니 실습: 지식 베이스에 새 정책 추가하고 질문해보기
각 단계에 한국어 설명 포함. 초보자도 전체 흐름을 이해할 수 있게 작성.
```

### 🇺🇸 English Prompt
```
Create a Korean Google Colab notebook implementing a simple RAG pipeline
using Anthropic Claude API + sentence-transformers. Include:
1. Install anthropic, sentence-transformers, numpy
2. Build a knowledge base: 6 paragraphs of fictional 'TechKorea' HR policy
   (vacation, remote work, benefits, equipment, expenses, training)
3. Generate embeddings with sentence-transformers, store as numpy arrays
4. Retrieval function: embed query → cosine similarity → return top-2 chunks
5. Augment prompt with retrieved chunks → Claude claude-sonnet-4-6 answer
6. Show 'without RAG vs with RAG' comparison for the same question
7. Mini exercise: add new policy text and ask about it
Korean comments throughout. Make the full RAG flow clear for beginners.
```

# FM1 실습 3: RAG 파이프라인 구현

## 학습 목표
- RAG(검색 증강 생성)의 전체 흐름을 직접 구현한다
- 임베딩 검색과 LLM을 결합하는 방법을 배운다
- RAG가 없을 때와 있을 때의 차이를 비교한다

## RAG 전체 흐름
```
[사전 작업]  문서 → 청크 분할 → 임베딩 → 벡터 저장
                                              ↓
[질문 시]    질문 → 임베딩 → 유사 청크 검색 → Claude에 전달 → 답변
```

In [ ]:
!pip install anthropic sentence-transformers numpy -q
print("✅ 설치 완료!")

In [ ]:
import getpass
import anthropic

api_key = getpass.getpass("🔑 Anthropic API 키를 입력하세요: ")
client = anthropic.Anthropic(api_key=api_key)
print("✅ 준비 완료!")

## 1단계: 지식 베이스 구성

실습용 가상 회사 'TechKorea'의 HR 정책 문서를 지식 베이스로 사용합니다.

In [ ]:
# 지식 베이스: 각 항목은 하나의 '문서 청크'입니다
knowledge_base = [
    {
        "id": "vacation",
        "title": "연차 휴가 정책",
        "content": "TechKorea 직원은 입사 1년 후 연 15일의 유급 연차를 부여받습니다. "
                   "연차는 매년 1월 1일 자동 지급되며, 당해 연도 미사용 연차의 50%는 "
                   "다음 연도로 이월할 수 있습니다. 연차 신청은 최소 3일 전에 HR 시스템을 통해 신청하며, "
                   "팀장의 승인 후 확정됩니다."
    },
    {
        "id": "remote",
        "title": "재택근무 정책",
        "content": "TechKorea는 주 3일 출근, 2일 재택근무를 기본 정책으로 합니다. "
                   "재택근무일은 팀별 협의로 지정하며, 화요일과 목요일을 공통 출근일로 권장합니다. "
                   "신입사원의 경우 입사 후 3개월간은 전일 출근을 원칙으로 합니다. "
                   "재택근무 시 오전 9시~오후 6시 코어타임을 준수해야 합니다."
    },
    {
        "id": "benefits",
        "title": "복리후생",
        "content": "TechKorea의 복리후생 프로그램: 중식 및 석식 무료 제공(구내식당), "
                   "건강검진 연 1회 지원(배우자 포함), 자녀 학자금 지원(중·고등학교), "
                   "사내 피트니스 센터 무료 이용, 명절 상품권 지급(설·추석), "
                   "장기근속 포상(5년·10년·15년 마일스톤 보상)."
    },
    {
        "id": "equipment",
        "title": "장비 지원 정책",
        "content": "신규 입사자에게 노트북(맥북 프로 또는 ThinkPad X1 선택), 모니터 27인치, "
                   "무선 키보드/마우스를 기본 지급합니다. 장비 교체 주기는 3년이며, "
                   "개인 업무 필요에 따라 추가 장비(패드, 외장 SSD 등)는 팀장 승인 후 신청 가능합니다. "
                   "재택근무 시 필요한 모니터 추가 1대는 신청일로부터 2주 내 배송됩니다."
    },
    {
        "id": "expense",
        "title": "경비 처리 정책",
        "content": "업무 관련 경비는 지출 후 30일 이내에 경비 시스템(ExPro)에 영수증 첨부하여 신청합니다. "
                   "식비: 1인 2만원 한도(고객 접대 시 5만원), 교통비: 실비 처리, "
                   "숙박비: 서울/수도권 15만원, 지방 10만원 한도. "
                   "5만원 초과 경비는 팀장 사전 승인이 필요합니다."
    },
    {
        "id": "training",
        "title": "교육 및 자기계발 지원",
        "content": "TechKorea는 직원의 역량 개발을 위해 연간 200만원의 교육비를 지원합니다. "
                   "지원 대상: 직무 관련 외부 교육, 자격증 취득, 도서 구매, 온라인 강의. "
                   "AI/데이터 관련 교육은 별도 예산으로 추가 지원합니다(연 100만원 추가). "
                   "교육 신청은 사내 포털에서 하며, HR의 승인 후 진행됩니다."
    }
]

print(f"✅ 지식 베이스 구성 완료: {len(knowledge_base)}개 정책 문서")
for item in knowledge_base:
    print(f"  - {item['title']}")

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 임베딩 모델 로드
print("임베딩 모델 로딩 중...")
embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 지식 베이스의 모든 문서를 임베딩으로 변환
texts = [item['content'] for item in knowledge_base]
kb_embeddings = embed_model.encode(texts)

print(f"✅ 임베딩 완료: {kb_embeddings.shape} (문서 수 × 벡터 차원)")

## 2단계: 검색 함수 구현

In [ ]:
def retrieve(query, kb_embeddings, knowledge_base, embed_model, top_k=2):
    """
    질문과 가장 관련 있는 문서 청크를 검색합니다.
    """
    # 질문을 임베딩으로 변환
    query_embedding = embed_model.encode([query])
    
    # 질문과 각 문서 간의 코사인 유사도 계산
    similarities = cosine_similarity(query_embedding, kb_embeddings)[0]
    
    # 유사도 높은 순으로 top_k개 선택
    top_indices = similarities.argsort()[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "title": knowledge_base[idx]['title'],
            "content": knowledge_base[idx]['content'],
            "similarity": similarities[idx]
        })
    return results

# 검색 테스트
test_query = "재택근무는 몇 일이나 가능한가요?"
results = retrieve(test_query, kb_embeddings, knowledge_base, embed_model)
print(f"질문: {test_query}")
print("\n검색된 문서:")
for r in results:
    print(f"  [{r['title']}] 유사도: {r['similarity']:.3f}")

## 3단계: RAG 답변 생성

In [ ]:
def answer_with_rag(query):
    """RAG를 사용하여 답변을 생성합니다."""
    # 1. 관련 문서 검색
    retrieved = retrieve(query, kb_embeddings, knowledge_base, embed_model)
    
    # 2. 검색된 문서를 컨텍스트로 구성
    context = "\n\n".join([
        f"[{r['title']}]\n{r['content']}" for r in retrieved
    ])
    
    # 3. 컨텍스트를 포함한 프롬프트 구성
    prompt = f"""다음은 TechKorea 회사의 HR 정책 문서입니다:

{context}

위 정책을 바탕으로 아래 질문에 정확하게 답변해주세요.
정책에 없는 내용은 "해당 정책 문서에 명시되어 있지 않습니다"라고 답하세요.

질문: {query}"""
    
    # 4. Claude로 답변 생성
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text


def answer_without_rag(query):
    """RAG 없이 Claude가 가진 지식만으로 답변합니다."""
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=256,
        messages=[{"role": "user", "content": query}]
    )
    return response.content[0].text

print("✅ 함수 준비 완료")

In [ ]:
# RAG 없음 vs RAG 있음 비교
question = "TechKorea 연차 휴가는 몇 일이고, 미사용 연차는 어떻게 되나요?"

print("=" * 60)
print(f"질문: {question}")
print("=" * 60)

print("\n❌ RAG 없이 (Claude 자체 지식만 사용):")
print("-" * 40)
print(answer_without_rag(question))

print("\n✅ RAG 사용 (회사 정책 문서 참조):")
print("-" * 40)
print(answer_with_rag(question))

print("\n💡 RAG가 있으면 회사 내부 문서를 기반으로 정확한 답변을 제공합니다!")

In [ ]:
# 추가 테스트 질문들
test_questions = [
    "재택근무는 주 몇 일 가능한가요?",
    "교육비 지원은 얼마나 받을 수 있나요?",
    "출장 숙박비는 어떻게 처리하나요?"
]

for q in test_questions:
    print(f"\n❓ {q}")
    print(f"💬 {answer_with_rag(q)}")
    print("-" * 50)

## ✏️ 미니 실습: 나만의 정책 추가하기

In [ ]:
# ✏️ 새로운 정책을 추가하고 질문해보세요!
new_policy = {
    "id": "my_policy",
    "title": "여기에 정책 제목을 입력하세요",
    "content": "여기에 정책 내용을 입력하세요. 예: 사내 동아리 활동비는 연 30만원까지 지원하며..."
}

# 지식 베이스와 임베딩에 추가
knowledge_base.append(new_policy)
new_embedding = embed_model.encode([new_policy['content']])
kb_embeddings = np.vstack([kb_embeddings, new_embedding])

# 새 정책에 대해 질문해보기
my_question = "새로 추가한 정책에 대해 질문을 입력하세요"
print(f"❓ {my_question}")
print(f"💬 {answer_with_rag(my_question)}")

## 📝 정리

| 단계 | 역할 | 사용 기술 |
|------|------|----------|
| 인덱싱 | 문서 → 임베딩 저장 | SentenceTransformer |
| 검색 | 질문 → 관련 문서 찾기 | 코사인 유사도 |
| 생성 | 문서 + 질문 → 답변 | Claude API |

**RAG의 핵심 장점:**
- 회사 내부 문서를 기반으로 정확한 답변
- Claude 재학습 없이 최신 정보 반영
- 출처 추적 가능 (어떤 문서에서 답변했는지)

**다음 강의 2에서:** Claude API로 챗봇, AI 에이전트 실습!